# Tracing GenAI con MLflow
Este ejemplo produce una traza padre-hijo sin consumir una API externa. Los spans aparecen en la pestaña **Traces**.

In [ ]:
import mlflow
from mlflow.entities import SpanType

mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("genai-tracing-notebook")

In [ ]:
@mlflow.trace(span_type=SpanType.RETRIEVER)
def retrieve_context(question: str) -> list[str]:
    documents = {
        "mlflow": "MLflow registra experimentos, modelos y trazas.",
        "uv": "uv crea entornos y bloquea dependencias reproducibles.",
    }
    return [text for topic, text in documents.items() if topic in question.lower()]


@mlflow.trace(span_type=SpanType.LLM, attributes={"model": "local-rule-based-demo"})
def generate_answer(question: str, context: list[str]) -> str:
    return " ".join(context) if context else f"Sin contexto para: {question}"


@mlflow.trace(name="answer-question", span_type=SpanType.CHAIN)
def answer_question(question: str) -> str:
    mlflow.update_current_trace(tags={"source": "notebook"})
    return generate_answer(question, retrieve_context(question))

In [ ]:
answer_question("¿Cómo se complementan MLflow y uv?")